# Team 3_1 Phase 2 Feature Engineering

## Imports and Setup

In [0]:
#Imports

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.stat import Correlation

from pyspark.sql.types import IntegerType, FloatType, DoubleType, LongType, ShortType, DecimalType
from pyspark.sql import functions as F
from pyspark.sql.functions import sum, expr, col, when, lit, regexp_extract, percentile_approx, regexp_replace, expr, count, to_timestamp, unix_timestamp, mean as _mean, stddev as _stddev

from pyspark.sql.window import Window

import holidays

import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
data_BASE_DIR = "dbfs:/mnt/mids-w261"
display(dbutils.fs.ls(f"{data_BASE_DIR}"))

In [0]:
 # OTPW
# df_otpw = spark.read.format("csv").option("header","true").load(f"{data_BASE_DIR}/OTPW_3M_2015.csv")
df_otpw = spark.read.format("csv").option("header","true").load(f"{data_BASE_DIR}/OTPW_12M/OTPW_12M/OTPW_12M_2015.csv.gz")
display(df_otpw.limit(10))

In [0]:
df_otpw.count()

In [0]:
df_otpw.columns


# ETL and Feature Engineering Pipeline

## Initial OTPW Dataframe

In [0]:
# Creating initial df for the feature engineering pipeline
df_otpw_engr = df_otpw.withColumn("FL_DATE", to_timestamp(col("FL_DATE"), "yyyy-MM-dd")).withColumn("DEP_DEL15", col("DEP_DEL15").cast("double").cast("boolean"))

# Drop rows where DEP_DEL15 is null
df_otpw_engr = df_otpw_engr.filter(col("DEP_DEL15").isNotNull())
df_otpw_engr = df_otpw_engr.cache()

print(df_otpw_engr.count())
display(df_otpw_engr.limit(10))

In [0]:
#check percentage ratio of DEP_DEL15
total_count = df_otpw_engr.count()

df_otpw_engr.groupBy("DEP_DEL15").count().withColumn("ratio", col("count") / total_count * 100).show()

### Tester Helper Functions

In [0]:
### This run_logistic_regression function will be used as a dummy function to ensure that the columns being selected as part of feature engineering are correctly selected for the model training

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from sklearn import metrics

def evaluate(lr_model, split_df):
    preds = lr_model.transform(split_df)
    
    # true_positive = preds.filter((col("prediction") == 1) & (col("label") == 1)).count()
    # false_positive = preds.filter((col("prediction") == 1) & (col("label") == 0)).count()
    # true_negative = preds.filter((col("prediction") == 0) & (col("label") == 0)).count()
    # false_negative = preds.filter((col("prediction") == 0) & (col("label") == 1)).count()

    # evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    # accuracy = evaluator.evaluate(preds)
    # recall_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="recallByLabel")
    # recall = recall_evaluator.evaluate(preds)
    # precision_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="precisionByLabel")
    # precision = precision_evaluator.evaluate(preds)

    unweighted_f1_macro = metrics.f1_score(
        y_true=preds.select("label").collect(),
        y_pred=preds.select("prediction").collect(),
        average="macro"
    )

    # predictionAndLabels = preds.select("prediction", "label").collect()
    # predictionAndLabels = sc.parallelize(predictionAndLabels)

    # metrics = BinaryClassificationMetrics(predictionAndLabels)

    # # F-beta score with beta=2.0 (F2 score)
    # f2_scores = metrics.fMeasureByThreshold(2.0)
    # f2_scores.collect()

    # Compute the unweighted (macro) F1 score
    # unweighted_f1_macro = evaluator_macro_sklearn.evaluate(predictions)
    # print(f"Unweighted (Macro) F1 Score = {unweighted_f1_macro}")

    # #display confusion matrix of the model
    # confusion_matrix = spark.createDataFrame([
    #     ("True Positive", true_positive),
    #     ("False Positive", false_positive),
    #     ("True Negative", true_negative),
    #     ("False Negative", false_negative)
    # ], ["label", "count"])
    # display(confusion_matrix)
    return unweighted_f1_macro

def run_logistic_regression(df, features=None, label_col="DEP_DEL15", test_ratio=0.2, seed=42):
    # Default features: all columns except label
    if features is None:
        features = [c for c in df.columns if c != label_col and c != "FL_DATE"]

        # Assemble features
        assembler = VectorAssembler(inputCols=features, outputCol="features", handleInvalid="skip")
        df = assembler.transform(df).withColumn("label", col(label_col).cast("double"))

    # # Split data
    # # Stratify by 'label_col' with desired ratios
    # fractions = {0: (1-test_ratio), 1: (1-test_ratio)}
    # train_df = df_model.sampleBy("DEP_DEL15", fractions, seed=seed)
    # test_df = df_model.subtract(train_df)

    train_df, test_df = df.randomSplit([1 - test_ratio, test_ratio], seed=seed)

    # Fit logistic regression
    lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
    lr_model = lr.fit(train_df)

    # Predict and evaluate
    print("Train")
    train_unweighted_f1 = evaluate(lr_model, train_df)
    print("Test")
    test_unweighted_f1 = evaluate(lr_model, test_df)

    metrics = spark.createDataFrame([
        ("train", train_unweighted_f1),
        ("test", test_unweighted_f1)
    ], ["split", "unweighted f1"])
    display(metrics)
    return lr_model, metrics


## Weather Column Selection in OTPW

### Column Selection

**Kept as they are (10 columns) :**

| Column                    | Why Keep                                                                 |
|---------------------------|--------------------------------------------------------------------------|
| HourlyVisibility          | Main cause of delays- low visibility leads to ground stops               |
| HourlyWindSpeed           | High winds can close runways                                             |
| HourlyWindGustSpeed       | Sudden gusts can force go-arounds                                        |
| HourlyWindDirection       | Crosswinds require runway changes, reducing capacity                     |
| HourlyPrecipitation       | Rain or snow slows ground operations and arrivals                        |
| HourlyDryBulbTemperature  | Extreme temperatures require de-icing or limit aircraft weight            |
| HourlyDewPointTemperature | Used to calculate temp-dewpoint spread (helps predict fog)               |
| HourlyRelativeHumidity    | High humidity plus cooling can cause fog or icing                        |
| HourlyPressureChange      | Rapid drops often mean storms are coming                                 |
| HourlySeaLevelPressure    | Low pressure usually means bad weather                                   |

**Transformed into engineered features (2 columns):**

| Column                    | Replaced By                                         | Feature Engineering                       |
|---------------------------|-----------------------------------------------------|-------------------------------------------|
| HourlySkyConditions       | ceiling_height_ft                                   | Extracted lowest BKN/OVC layer altitude   |
| HourlyPresentWeatherType  | has_thunder, has_rain, has_snow, has_fog, etc.      | Parsed into model-ready binary flags      |

**Dropped (4 columns):**

| Column                    | Why Drop                                                                 |
|---------------------------|--------------------------------------------------------------------------|
| HourlyAltimeterSetting    | Has multicollinearity with  HourlySeaLevelPressure.                      |
| HourlyStationPressure     | Same as sea level pressure but just shifted by elevation-redundant       |
| HourlyWetBulbTemperature  | Can be calculated from DryBulb and Humidity-no extra information, also has high collinearity with HourlyDewPointTemperature and  HourlyDryBulbTemperature.     |
| HourlyPressureTendency    | Just a coarse version of HourlyPressureChange, which is already included |

In [0]:
weather_cols_string = [
 "HourlyPresentWeatherType", #-> we can create binary columns like thunderstorm,rain,fog,snow from this column.
"HourlySkyConditions", #-> BKN (Broken) and OVC (Overcast) are cloud cover categories
]
weather_cols_categorical = []

weather_cols_numeric = [
    "HourlyVisibility",
    "HourlyWindSpeed",
    "HourlyWindGustSpeed",
    "HourlyWindDirection",
    "HourlyPrecipitation",
    "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature",
    "HourlyRelativeHumidity",
    "HourlyPressureChange",
    "HourlySeaLevelPressure",
    "HourlyPressureTendency",
]


### Feature Engineering

In [0]:
selected_weather_columns = weather_cols_categorical + weather_cols_numeric + weather_cols_string

In [0]:
df_otpw_engr_weather = df_otpw_engr.select(selected_weather_columns + ["MONTH", "FL_DATE", "DEP_DEL15"])
display(df_otpw_engr_weather)
df_otpw_engr_weather.select('DEP_DEL15').count()


In [0]:

# --- 1. Ceiling height from HourlySkyConditions ---

def weather_feature_engr(df):

    def parse_ceiling(sky):
        """Extract lowest BKN/OVC/VV layer altitude in feet."""
        if sky is None:
            return None
        # Match BKN, OVC, or VV layers and capture the height number that follows
        matches = re.findall(r'(?:BKN|OVC|VV):\d+\s+(\d+)', sky)
        if not matches:
            return 99999  # no ceiling layer → clear or only FEW/SCT
        return min(int(h) * 100 for h in matches)

    parse_ceiling_udf = udf(parse_ceiling, IntegerType())

    # --- 2. Weather-type binary flags from HourlyPresentWeatherType ---

    def has_code(col_name, code):
        """Return 1 if *code* appears anywhere in the string, 0 otherwise."""
        return (
            when(col(col_name).isNull(), 0)
            .when(col(col_name).contains(code), 1)
            .otherwise(0)
        )

    # --- 3. Apply all transformations to df ---

    df = (
        df
        # Ceiling height from sky conditions
        .withColumn("ceiling_height_ft", parse_ceiling_udf(col("HourlySkyConditions")))
        # Binary weather-event flags
        .withColumn("has_thunder",  has_code("HourlyPresentWeatherType", "TS"))
        .withColumn("has_rain",     has_code("HourlyPresentWeatherType", "RA"))
        .withColumn("has_snow",     has_code("HourlyPresentWeatherType", "SN"))
        .withColumn("has_fog",      has_code("HourlyPresentWeatherType", "FG"))
        .withColumn("has_freezing", has_code("HourlyPresentWeatherType", "FZ"))
        .withColumn("has_haze",     has_code("HourlyPresentWeatherType", "HZ"))
    )

    # --- 4. Define list of created columns for downstream cells  ---

    created_weather_columns = [
        "ceiling_height_ft",  # Lowest BKN/OVC/VV cloud-layer altitude (feet)
        "has_thunder",        # 1 if thunderstorm (TS) reported
        "has_rain",           # 1 if rain (RA) reported
        "has_snow",           # 1 if snow (SN) reported
        "has_fog",            # 1 if fog (FG) reported
        "has_freezing",       # 1 if freezing precipitation (FZ) reported
        "has_haze",           # 1 if haze (HZ) reported
    ]

    # Quick verification — show rows that have weather events
    print(f"Created {len(created_weather_columns)} new weather features from string columns.")
    display(
        df
        .filter(col("HourlyPresentWeatherType").isNotNull())
        .select(
            "HourlySkyConditions", "ceiling_height_ft",
            "HourlyPresentWeatherType",
            "has_thunder", "has_rain", "has_snow",
            "has_fog", "has_freezing", "has_haze",
        )
        .limit(20)
    )
    return df

In [0]:
df_otpw_engr_weather = weather_feature_engr(df_otpw_engr_weather)

### Clean and Cast Weather Columns

In [0]:

def cleanWeatherColumns(df_otpw_engr_weather, weather_cols_numeric = weather_cols_numeric, weather_cols_categorical = weather_cols_categorical):
    # Clean columns
    # Known non-numeric patterns in NOAA weather data:
    #   '*' : quality flag (trailing e.g. '29.94*', or standalone -> null)
    #   's' : suspect value suffix (e.g. '46s')
    #   'T' : trace precipitation (≈ 0, treated as 0)
    #   'VRB' : variable wind direction (treated as null)

    for c in weather_cols_numeric:
        df_otpw_engr_weather = (
            df_otpw_engr_weather
            # standalone *, VRB -> null; T (trace precip) -> '0'
            .withColumn(c, when(col(c) == '*', lit(None))
                        .when(col(c) == 'T', lit('0'))
                        .when(col(c) == 'VRB', lit(None))
                        .otherwise(col(c)))
            # strip trailing * and s flags 
            .withColumn(c, regexp_replace(col(c), r'[*s]+$', ''))
            # extract numeric part
            .withColumn(c, regexp_extract(col(c), r'([+-]?\d+\.?\d*)', 1))
            # empty -> null
            .withColumn(c, when(col(c) == '', lit(None)).otherwise(col(c)))
            # cast to double
            # .withColumn(c, col(c).cast('double'))
            .withColumn(c, expr(f"try_cast(`{c}` as double)"))
        )


    # Verify null counts 
    total = df_otpw_engr_weather.count()
    non_null_counts = df_otpw_engr_weather.select(
        *[count(when(col(c).isNotNull(), 1)).alias(c) for c in weather_cols_numeric]
    ).toPandas().T.reset_index()
    non_null_counts.columns = ['column', 'non_null']
    non_null_counts['null'] = total - non_null_counts['non_null']
    non_null_counts['null_pct'] = (non_null_counts['null'] / total * 100).round(1)

    print(f"Total rows: {total:,}\n")
    # display(non_null_counts)
    # df_otpw_engr_weather.printSchema()
    return df_otpw_engr_weather



In [0]:
df_otpw_engr_weather = cleanWeatherColumns(df_otpw_engr_weather)

### Data Imputation

In [0]:
# Data Imputation

def impute_weather_columns(df):
    """
    Impute nulls in weather columns.
    Three tiers:
      1. Zero-fill   -> columns where null = "nothing noteworthy happened"
      2. Domain fill  -> ceiling_height_ft null = clear sky (99999)
      3. Median fill  -> slow-moving continuous variables with low null rates
    """

    # ---- 1. Zero-fill: null means the phenomenon wasn't observed ----
    zero_fill_cols = {
        "HourlyWindGustSpeed": 0,   # gusts only reported above ~14 kt
        "HourlyPrecipitation": 0,   # blank = dry conditions
        "HourlyPressureChange": 0,  # blank = stable pressure
        "HourlyWindDirection": 0,   # blank = calm / no direction
    }
    for c, fill_val in zero_fill_cols.items():
        if c in df.columns:
            df = df.withColumn(c, when(col(c).isNull(), lit(fill_val)).otherwise(col(c)))

    # ---- 2. Domain default: ceiling_height_ft ----
    # No sky-condition report -> assume unlimited ceiling (clear sky)
    if "ceiling_height_ft" in df.columns:
        df = df.withColumn(
            "ceiling_height_ft",
            when(col("ceiling_height_ft").isNull(), lit(99999))
            .otherwise(col("ceiling_height_ft"))
        )

    # ---- 3. Median fill: continuous measurements with low null rates ----
    # These are slow-moving physical quantities. Median is a safe to use.
    median_fill_cols = [
        "HourlySeaLevelPressure",    # 9.6% null
        "HourlyVisibility",          # 0.25%
        "HourlyWindSpeed",           # 0.29%
        "HourlyDryBulbTemperature",  # 0.26%
        "HourlyDewPointTemperature", # 0.27%
        "HourlyRelativeHumidity",    # 0.28%
        "HourlyPressureTendency"
    ]
    existing = [c for c in median_fill_cols if c in df.columns]

    if existing:
        # Compute approximate medians 
        medians_row = df.select(
            *[percentile_approx(col(c), 0.5).alias(c) for c in existing]
        ).collect()[0]

        for c in existing:
            med_val = medians_row[c]
            if med_val is not None:
                df = df.withColumn(c, when(col(c).isNull(), lit(med_val)).otherwise(col(c)))
                print(f"  {c}: filled nulls with median = {med_val}")

    return df


In [0]:

print("Imputing weather columns...\n")
df_otpw_engr_weather = impute_weather_columns(df_otpw_engr_weather)
print("\nDone.")

In [0]:
# Null Count Verification

# Confirm all weather columns are now null-free
from pyspark.sql.functions import col, count, when, round as spark_round

all_cols = [c for c in df_otpw_engr_weather.columns if c not in ("FL_DATE", "MONTH", "DEP_DEL15")]
total = df_otpw_engr_weather.count()

post_null = df_otpw_engr_weather.select(
    *[spark_round(count(when(col(c).isNull(), 1)) / total * 100, 2).alias(c) for c in all_cols]
)

import pandas as pd
post_df = post_null.toPandas().T.reset_index()
post_df.columns = ["column", "null_pct_after"]
post_df = post_df.sort_values("null_pct_after", ascending=False)

print(f"Total rows: {total:,}")
print(f"Columns with remaining nulls: {(post_df['null_pct_after'] > 0).sum()}\n")
display(post_df)

### Modeling Readiness

In [0]:
numeric_cols = [c for c in df_otpw_engr_weather.columns if c != "DEP_DEL15" and c != "FL_DATE" and dict(df_otpw_engr_weather.dtypes)[c] != 'string']
numeric_cols

In [0]:
# Normalization of the numeric features of the df_engr_mini_sample dataset using StandardScaler
df_corr = df_otpw_engr_weather.select(*(numeric_cols+["FL_DATE", "MONTH", "DEP_DEL15"])).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vec_w = assembler.transform(df_corr).withColumn("label", col("DEP_DEL15").cast("double"))

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model_w = scaler.fit(df_vec_w)
df_scaled_weather = scaler_model_w.transform(df_vec_w) #.select("scaled_features")

# Assemble features

# df_scaled_fin = df_scaled.drop("features") #I want to overwrite the features as it is already scaled in scaled_features
# feature_cols = ["YEAR", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "QUARTER", "CRS_DEP_TIME", "CRS_ARR_TIME", "scaled_features"]
# assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
# df_model = assembler.transform(df_scaled_fin).withColumn("label", col("DEP_DEL15").cast("double"))

df_scaled_weather = df_scaled_weather.select("FL_DATE", "MONTH", "scaled_features", "label")


In [0]:
display(df_scaled_weather)

In [0]:
corr_matrix = Correlation.corr(df_scaled_weather, "scaled_features", "pearson") \
                         .collect()[0][0].toArray()

corr_df = pd.DataFrame(corr_matrix, columns=numeric_cols, index=numeric_cols)

#Plot correlation matrix

plt.figure(figsize=(10,8))
sns.heatmap(corr_df, annot=False, xticklabels=numeric_cols, yticklabels=numeric_cols, cmap="coolwarm")
plt.title("Correlation Matrix of Standardized Numeric Features")
plt.show()
plt.savefig("/tmp/FinalFeatureEngineeringCorrelationMatrixWeather.png")

In [0]:
# Rerun weather standardization after dropping collineated variables
df_otpw_engr_weather = df_otpw_engr_weather.drop("HourlyDewPointTemperature", "HourlyPressureTendency")


# Normalization of the numeric features of the df_engr_mini_sample dataset using StandardScaler
df_corr = df_otpw_engr_weather.select(*(numeric_cols+["FL_DATE", "MONTH", "DEP_DEL15"])).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vec_w = assembler.transform(df_corr).withColumn("label", col("DEP_DEL15").cast("double"))

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model_w = scaler.fit(df_vec_w)
df_scaled_weather = scaler_model_w.transform(df_vec_w) #.select("scaled_features")

# Assemble features

# df_scaled_fin = df_scaled.drop("features") #I want to overwrite the features as it is already scaled in scaled_features
# feature_cols = ["YEAR", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "QUARTER", "CRS_DEP_TIME", "CRS_ARR_TIME", "scaled_features"]
# assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
# df_model = assembler.transform(df_scaled_fin).withColumn("label", col("DEP_DEL15").cast("double"))

df_scaled_weather = df_scaled_weather.select("FL_DATE", "MONTH", "scaled_features", "label")


In [0]:
# Rerun weather standardization after dropping collineated variables
df_otpw_engr_weather = df_otpw_engr_weather.drop("HourlyDewPointTemperature", "HourlyPressureTendency")


# Normalization of the numeric features of the df_engr_mini_sample dataset using StandardScaler
df_corr = df_otpw_engr_weather.select(*(numeric_cols+["FL_DATE", "MONTH", "DEP_DEL15"])).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vec_w = assembler.transform(df_corr).withColumn("label", col("DEP_DEL15").cast("double"))

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model_w = scaler.fit(df_vec_w)
df_scaled_weather = scaler_model_w.transform(df_vec_w) #.select("scaled_features")

# Assemble features

# df_scaled_fin = df_scaled.drop("features") #I want to overwrite the features as it is already scaled in scaled_features
# feature_cols = ["YEAR", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "QUARTER", "CRS_DEP_TIME", "CRS_ARR_TIME", "scaled_features"]
# assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
# df_model = assembler.transform(df_scaled_fin).withColumn("label", col("DEP_DEL15").cast("double"))

df_scaled_weather = df_scaled_weather.select("FL_DATE", "MONTH", "scaled_features", "label")


In [0]:
df_scaled_weather = df_scaled_weather.withColumnRenamed("scaled_features", "features")

### Logistic Regression Dummy Model Tester
Ensuring that the columns can be used in a logistic regression model

In [0]:
weather_model, weather_metrics = run_logistic_regression(df_scaled_weather, features="features", label_col="label", seed=42)

In [0]:
# Checkpoint Weather Data
section = "3"
number = "1"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"
dbutils.fs.mkdirs(folder_path)

# Save df_etl as a parquet file
df_scaled_weather.write.mode('overwrite').parquet(f"{folder_path}/delay_df_phase_2_weather.parquet")

## Flights Column Selection in OTPW

### Column Selection

In [0]:
flights_cols_string = [
    "OP_UNIQUE_CARRIER",    #yes
    "ORIGIN",               #yes
    "DEST",                 #yes
    "ORIGIN_STATE_ABR",     #yes
    "DEST_STATE_ABR",       #yes
    "TAIL_NUM",             #yes
    "OP_CARRIER_FL_NUM",
]

flights_cols_categorical = [
    "QUARTER",             # Seasonal delay patterns
    "MONTH",               # Monthly delay variation
    "DAY_OF_MONTH",        # Day-of-month patterns
    "DAY_OF_WEEK",         # Day-of-week delay variation
    "YEAR",                # Year effects for multi-year datasets
    "CRS_DEP_TIME",        # Scheduled departure time — strong intra-day signal
    "CRS_ARR_TIME",        # Scheduled arrival time
]
# FL_DATE, 

flights_cols_numeric = [
    "DISTANCE",            # Route distance
]

In [0]:
selected_flights_columns = flights_cols_categorical + flights_cols_numeric + flights_cols_string

In [0]:
df_otpw_engr_flights = df_otpw_engr[selected_flights_columns + ["FL_DATE", "DEP_DEL15"]]
df_otpw_engr_flights = df_otpw_engr_flights.cache()
display(df_otpw_engr_flights.head(10))

### Clean and Cast Flights Columns

In [0]:
def cleanFlightsColumns(df_otpw_engr_flights, flights_cols_numeric=flights_cols_numeric, flights_cols_categorical=flights_cols_categorical, flights_cols_string=flights_cols_string):
    # Clean and cast numeric flight columns
    for c in flights_cols_numeric:
        if c in df_otpw_engr_flights.columns:
            df_otpw_engr_flights = df_otpw_engr_flights.withColumn(c, expr(f"try_cast(`{c}` as double)"))

    # Clean and cast categorical flight columns
    for c in flights_cols_categorical:
        if c in df_otpw_engr_flights.columns:
            df_otpw_engr_flights = df_otpw_engr_flights.withColumn(c, expr(f"try_cast(`{c}` as int)"))

    # Drop rows where any selected flight column is null
    critical_cols = [
        "OP_UNIQUE_CARRIER", "ORIGIN",
        "QUARTER", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "CRS_DEP_TIME", "YEAR"
    ]
    df_otpw_engr_flights = df_otpw_engr_flights.dropna(subset=[c for c in critical_cols if c in df_otpw_engr_flights.columns])

    display(df_otpw_engr_flights)
    df_otpw_engr_flights.printSchema()
    print(df_otpw_engr_flights.count())
    return df_otpw_engr_flights


In [0]:
df_otpw_engr_flights = cleanFlightsColumns(df_otpw_engr_flights)
df_otpw_engr_flights = df_otpw_engr_flights.cache() 

### Feature Engineered Columns for Flights

In [0]:
def featureEngineeredFlights(df):

    ## Order of current flight in today's schedule in airport X

    window_day = Window.partitionBy("FL_DATE").orderBy("CRS_DEP_TIME")

    df = df.withColumn(
        "departure_sequence",
        F.row_number().over(window_day)
    )

    ## Accumulated scheduled flights for today

    window_cum_flights = (
        Window.partitionBy("ORIGIN", "FL_DATE")
            .orderBy("CRS_DEP_TIME", "OP_CARRIER_FL_NUM")
            .rowsBetween(Window.unboundedPreceding, -1)
    )

    df = df.withColumn(
        "scheduled_flights_before_current",
        F.count(F.lit(1)).over(window_cum_flights)
    )

    df = df.fillna(
        {"scheduled_flights_before_current": 0}
    )
    print("Created scheduled_flights_before_current")

    ## Early-morning delay propagation

    # Extract hour from HHMM format
    df = df.withColumn(
        "dep_hour",
        (F.col("CRS_DEP_TIME") / 100).cast("int")
    )

    # Early morning flag
    df = df.withColumn(
        "is_early_morning",
        F.col("dep_hour") < 8
    )

    morning_rates = (
        df.filter(F.col("is_early_morning"))
        .groupBy("ORIGIN", "FL_DATE")
        .agg(
            F.avg(F.col("DEP_DEL15").cast("int")).alias("morning_delay_rate_airport_day")
        )
    )

    # Historical average morning delay rate by airport
    airport_avg_morning_delay = (
        morning_rates
        .groupBy("ORIGIN")
        .agg(F.avg("morning_delay_rate_airport_day").alias("avg_morning_delay_rate_airport"))
    )

    # Add airport benchmark to morning rates
    morning_rates = morning_rates.join(
        airport_avg_morning_delay,
        on="ORIGIN",
        how="left"
    )

    # Join back to full dataset
    df = df.join(
        morning_rates.select(
            "ORIGIN",
            "FL_DATE",
            "morning_delay_rate_airport_day",
            "avg_morning_delay_rate_airport"
        ),
        on=["ORIGIN", "FL_DATE"],
        how="left"
    )

    # Final classification
    df = df.withColumn(
        "morning_delay_propagation",
        F.when(F.col("is_early_morning"), F.lit("early-morning flight"))
        .when(F.col("morning_delay_rate_airport_day").isNull(), F.lit(None))
        .when(
            F.col("morning_delay_rate_airport_day") > F.col("avg_morning_delay_rate_airport"),
            F.lit("high-delay morning")
        )
        .otherwise(F.lit("low-delay morning"))
    )

    df = df.drop("morning_delay_rate_airport_day","avg_morning_delay_rate_airport",
                                                    "dep_hour", "is_early_morning")
    
    ## Delay accumulation within the same day

    window_cum = Window.partitionBy("ORIGIN", "FL_DATE") \
                    .orderBy("CRS_DEP_TIME") \
                    .rowsBetween(Window.unboundedPreceding, -1)

    df = df.withColumn(
        "cum_delay_rate_today",
        F.avg(F.col("DEP_DEL15").cast('int')).over(window_cum)
    )

    print("Created cum_delay_rate_today")

    ## Yesterday's delay propagation

    # Daily delay rate by airport
    airport_daily_delay = (
        df.groupBy("ORIGIN", "FL_DATE")
        .agg(
            F.avg(F.col("DEP_DEL15").cast("int")).alias("airport_daily_delay_rate")
        )
    )

    # Historical average delay rate for each airport
    airport_historical_avg = (
        airport_daily_delay
        .groupBy("ORIGIN")
        .agg(F.avg("airport_daily_delay_rate").alias("airport_historical_avg_delay"))
    )

    # Join airport historical average
    airport_daily_delay = airport_daily_delay.join(
        airport_historical_avg,
        on="ORIGIN",
        how="left"
    )

    # Get yesterday's delay rate for each airport
    window_spec = Window.partitionBy("ORIGIN").orderBy("FL_DATE")

    airport_daily_delay = airport_daily_delay.withColumn(
        "yesterday_delay_rate",
        F.lag("airport_daily_delay_rate").over(window_spec)
    )

    # Classify yesterday vs that airport's historical average
    airport_daily_delay = airport_daily_delay.withColumn(
        "day_before_delay_propagation",
        F.when(F.col("yesterday_delay_rate").isNull(), F.lit(None))
        .when(
            F.col("yesterday_delay_rate") > F.col("airport_historical_avg_delay"),
            F.lit("high-delay yesterday")
        )
        .otherwise(F.lit("low-delay yesterday"))
    )

    # Join feature back to original dataset
    df = df.join(
        #airport_daily_delay.select("ORIGIN", "FL_DATE", "day_before_delay_propagation"),
        airport_daily_delay.select("ORIGIN", "FL_DATE", "day_before_delay_propagation"),
        on=["ORIGIN", "FL_DATE"],
        how="left"
    )

    print("Created day_before_delay_propagation")

    ## 7-day delay propagation

    # Daily delay rate by airport
    airport_daily_delay = (
        df
        .groupBy("ORIGIN", "FL_DATE")
        .agg(
            F.avg(F.col("DEP_DEL15").cast("int")).alias("airport_daily_delay_rate")
        )
    )

    # Historical average delay rate for each airport
    airport_historical_avg = (
        airport_daily_delay
        .groupBy("ORIGIN")
        .agg(
            F.avg("airport_daily_delay_rate").alias("airport_historical_avg_delay")
        )
    )

    # Join airport historical average
    airport_daily_delay = airport_daily_delay.join(
        airport_historical_avg,
        on="ORIGIN",
        how="left"
    )

    # Window for previous 7 days by airport
    window_7d = (
        Window.partitionBy("ORIGIN")
        .orderBy("FL_DATE")
        .rowsBetween(-7, -1)
    )

    # Compute average delay rate over previous 7 days
    airport_daily_delay = airport_daily_delay.withColumn(
        "previous_7d_delay_rate",
        F.avg("airport_daily_delay_rate").over(window_7d)
    )

    # Classify previous 7 days vs airport historical average
    airport_daily_delay = airport_daily_delay.withColumn(
        "7_day_delay_propagation",
        F.when(F.col("previous_7d_delay_rate").isNull(), F.lit(None))
        .when(
            F.col("previous_7d_delay_rate") > F.col("airport_historical_avg_delay"),
            F.lit("high-delay previous 7d")
        )
        .otherwise(F.lit("low-delay previous 7d"))
    )

    # Join feature back to original dataset
    df = df.join(
        airport_daily_delay.select("ORIGIN", "FL_DATE", "7_day_delay_propagation"),
        on=["ORIGIN", "FL_DATE"],
        how="left"
    )

    print("Created previous_7d_delay_rate")

    ## 15-day delay propagation

    days = 15

    prev_delay_col = f"previous_{days}d_delay_rate"
    propagation_col = f"{days}_day_delay_propagation"

    # Daily delay rate by airport
    airport_daily_delay = (
        df
        .groupBy("ORIGIN", "FL_DATE")
        .agg(
            F.avg(F.col("DEP_DEL15").cast("int")).alias("airport_daily_delay_rate")
        )
    )

    # Historical average delay rate for each airport
    airport_historical_avg = (
        airport_daily_delay
        .groupBy("ORIGIN")
        .agg(
            F.avg("airport_daily_delay_rate").alias("airport_historical_avg_delay")
        )
    )

    # Join airport historical average
    airport_daily_delay = airport_daily_delay.join(
        airport_historical_avg,
        on="ORIGIN",
        how="left"
    )

    # Window for previous X days by airport
    window_xd = (
        Window.partitionBy("ORIGIN")
        .orderBy("FL_DATE")
        .rowsBetween(-days, -1)
    )

    # Compute average delay rate over previous X days
    airport_daily_delay = airport_daily_delay.withColumn(
        prev_delay_col,
        F.avg("airport_daily_delay_rate").over(window_xd)
    )

    # Classify previous X days vs airport historical average
    airport_daily_delay = airport_daily_delay.withColumn(
        propagation_col,
        F.when(F.col(prev_delay_col).isNull(), F.lit(None))
        .when(
            F.col(prev_delay_col) > F.col("airport_historical_avg_delay"),
            F.lit(f"high-delay previous {days}d")
        )
        .otherwise(F.lit(f"low-delay previous {days}d"))
    )

    # Join feature back to original dataset
    df = df.join(
        airport_daily_delay.select("ORIGIN", "FL_DATE", propagation_col),
        on=["ORIGIN", "FL_DATE"],
        how="left"
    )

    print("Created 15_day_delay_propagation")

    ## 30-day delay propagation

    days = 30

    prev_delay_col = f"previous_{days}d_delay_rate"
    propagation_col = f"{days}_day_delay_propagation"

    # Daily delay rate by airport
    airport_daily_delay = (
        df
        .groupBy("ORIGIN", "FL_DATE")
        .agg(
            F.avg(F.col("DEP_DEL15").cast("int")).alias("airport_daily_delay_rate")
        )
    )

    # Historical average delay rate for each airport
    airport_historical_avg = (
        airport_daily_delay
        .groupBy("ORIGIN")
        .agg(
            F.avg("airport_daily_delay_rate").alias("airport_historical_avg_delay")
        )
    )

    # Join airport historical average
    airport_daily_delay = airport_daily_delay.join(
        airport_historical_avg,
        on="ORIGIN",
        how="left"
    )

    # Window for previous X days by airport
    window_xd = (
        Window.partitionBy("ORIGIN")
        .orderBy("FL_DATE")
        .rowsBetween(-days, -1)
    )

    # Compute average delay rate over previous X days
    airport_daily_delay = airport_daily_delay.withColumn(
        prev_delay_col,
        F.avg("airport_daily_delay_rate").over(window_xd)
    )

    # Classify previous X days vs airport historical average
    airport_daily_delay = airport_daily_delay.withColumn(
        propagation_col,
        F.when(F.col(prev_delay_col).isNull(), F.lit(None))
        .when(
            F.col(prev_delay_col) > F.col("airport_historical_avg_delay"),
            F.lit(f"high-delay previous {days}d")
        )
        .otherwise(F.lit(f"low-delay previous {days}d"))
    )

    # Join feature back to original dataset
    df = df.join(
        airport_daily_delay.select("ORIGIN", "FL_DATE", propagation_col),
        on=["ORIGIN", "FL_DATE"],
        how="left"
    )

    print("Created 30_day_delay_propagation")

    ## Holiday indicator variable

    # US holidays
    years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
    us_holidays = holidays.US(years=years)

    # Convert to list of strings (same format as FL_DATE)
    holiday_dates = [str(date) for date in us_holidays.keys()]

    df = df.withColumn(
        "is_us_holiday",
        F.when(F.col("FL_DATE").isin(holiday_dates), 1).otherwise(0)
    )

    print("Created is_us_holiday")

    ## Holiday proximity

    holidays_df = spark.createDataFrame(
        [(d,) for d in holiday_dates],
        ["holiday_date"]
    )

    #df = df.withColumn("FL_DATE", F.to_date("FL_DATE"))

    # Compute all the distances between each date to each holiday
    df_cross = df.crossJoin(F.broadcast(holidays_df))
    # Compute absolute differences in days
    df_cross = df_cross.withColumn(
        "days_diff",
        F.abs(F.datediff(F.col("FL_DATE"), F.col("holiday_date")))
    )
    # Get nearest holiday
    holiday_proximity = (
        df_cross.groupBy("FL_DATE")
                .agg(F.min("days_diff").alias("days_to_nearest_holiday"))
    )

    df = df.join(
        holiday_proximity.select("FL_DATE", "days_to_nearest_holiday"),
        on=["FL_DATE"],
        how="left"
    )

    print("Created days_to_nearest_holiday")

    ## Holiday proximity buckets

    df = df.withColumn(
        "holiday_proximity_bucket",
        F.when(F.col("days_to_nearest_holiday") == 0, "holiday")
        .when(F.col("days_to_nearest_holiday") <= 1, "±1 day")
        .when(F.col("days_to_nearest_holiday") <= 3, "±3 days")
        .when(F.col("days_to_nearest_holiday") <= 7, "±1 week")
        .otherwise("normal")
    )

    print("Created holiday_proximity_bucket")


    ## Fill nulls before encoding categorical engineered features

    categorical_cols = [
        "ORIGIN",
        "DEST",
        "ORIGIN_STATE_ABR",
        "DEST_STATE_ABR",
        "morning_delay_propagation",
        "day_before_delay_propagation",
        "7_day_delay_propagation",
        "15_day_delay_propagation",
        "30_day_delay_propagation",
        "holiday_proximity_bucket"
    ]

    numeric_fill_map = {
        "scheduled_flights_before_current": 0,
        "cum_delay_rate_today": 0.0,
        "days_to_nearest_holiday": 999
    }

    categorical_fill_map = {c: "missing" for c in categorical_cols}

    df = df.fillna(numeric_fill_map)
    df = df.fillna(categorical_fill_map)

    print("Filled up null values")

    ## Encode categorical engineered features
    ## Output:
    ## - *_idx  : indexed numeric category
    ## - *_ohe  : one-hot encoded vector for modeling

    indexed_cols = [f"{c}_idx" for c in categorical_cols]
    ohe_cols = [f"{c}_ohe" for c in categorical_cols]

    for c in categorical_cols:
        indexer = StringIndexer(
            inputCol=c,
            outputCol=f"{c}_idx",
            handleInvalid="keep"
        )
        df = indexer.fit(df).transform(df)

    encoder = OneHotEncoder(
        inputCols=indexed_cols,
        outputCols=ohe_cols,
        handleInvalid="keep"
    )

    df = encoder.fit(df).transform(df)

    print("One-hot encoded features")
    
    df = df.drop(*indexed_cols)

    return df


In [0]:
df_otpw_engr_flights = featureEngineeredFlights(df_otpw_engr_flights).cache()

In [0]:
df_otpw_engr_flights.printSchema()

In [0]:
# numeric_types = (IntegerType, DoubleType, FloatType, LongType, ShortType, DecimalType)

# numeric_cols = [
#     field.name
#     for field in df_otpw_engr_flights_new.schema.fields
#     if isinstance(field.dataType, numeric_types)
# ]
# ohe_columns = [col for col in df_otpw_engr_flights_new.columns if "ohe" in col]

In [0]:
display(df_otpw_engr_flights.limit(5))

In [0]:
# Null value checks for flights data
null_counts = df_otpw_engr_flights.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_otpw_engr_flights.columns
])

display(null_counts)

## Model Readiness

In [0]:
numeric_cols = [c for c in df_otpw_engr_flights.columns if c != "DEP_DEL15" and c != "FL_DATE" and dict(df_otpw_engr_flights.dtypes)[c] != 'string']
numeric_cols

In [0]:
# Normalization of the numeric features of the df_engr_mini_sample dataset using StandardScaler
df_corr = df_otpw_engr_flights.select(*(numeric_cols+["FL_DATE", "DEP_DEL15"])).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vec_w = assembler.transform(df_corr).withColumn("label", col("DEP_DEL15").cast("double"))

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model_w = scaler.fit(df_vec_w)
df_scaled_flights = scaler_model_w.transform(df_vec_w) #.select("scaled_features")

# Assemble features

# df_scaled_fin = df_scaled.drop("features") #I want to overwrite the features as it is already scaled in scaled_features
# feature_cols = ["YEAR", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "QUARTER", "CRS_DEP_TIME", "CRS_ARR_TIME", "scaled_features"]
# assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
# df_model = assembler.transform(df_scaled_fin).withColumn("label", col("DEP_DEL15").cast("double"))

df_scaled_flights =df_scaled_flights.select("FL_DATE", "MONTH", "scaled_features", "label")


In [0]:
display(df_scaled_flights, limit = 10)

In [0]:
corr_matrix = Correlation.corr(df_scaled_flights, "scaled_features", "pearson") \
                         .collect()[0][0].toArray()

corr_df = pd.DataFrame(corr_matrix, columns=numeric_cols, index=numeric_cols)

#Plot correlation matrix

plt.figure(figsize=(10,8))
sns.heatmap(corr_df, annot=False, xticklabels=numeric_cols, yticklabels=numeric_cols, cmap="coolwarm")
plt.title("Correlation Matrix of Standardized Numeric Features")
plt.show()
plt.savefig("/tmp/FinalFeatureEngineeringCorrelationMatrixFlights.png")

In [0]:
# Rerun flights standardization after dropping collineated variables
df_otpw_engr_flights = df_otpw_engr_flights.drop("HourlyDewPointTemperature", "HourlyPressureTendency")


# # Normalization of the numeric features of the df_engr_mini_sample dataset using StandardScaler
# df_corr = df_otpw_engr_flights.select(*(numeric_cols+["FL_DATE", "MONTH", "DEP_DEL15"])).dropna()

# assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
# df_vec_w = assembler.transform(df_corr).withColumn("label", col("DEP_DEL15").cast("double"))

# scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
# scaler_model_w = scaler.fit(df_vec_w)
# df_scaled_flights = scaler_model_w.transform(df_vec_w) #.select("scaled_features")

# # Assemble features

# # df_scaled_fin = df_scaled.drop("features") #I want to overwrite the features as it is already scaled in scaled_features
# # feature_cols = ["YEAR", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "QUARTER", "CRS_DEP_TIME", "CRS_ARR_TIME", "scaled_features"]
# # assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
# # df_model = assembler.transform(df_scaled_fin).withColumn("label", col("DEP_DEL15").cast("double"))

# df_scaled_flights = df_scaled_flights.select("FL_DATE", "MONTH", "scaled_features", "label")


In [0]:
df_scaled_flights = df_scaled_flights.withColumnRenamed("scaled_features", "features")

In [0]:
#Checkpoint Data
# Checkpoint Flights Data
section = "3"
number = "1"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"
dbutils.fs.mkdirs(folder_path)


# Save df_etl as a parquet file
df_scaled_flights.write.mode('overwrite').parquet(f"{folder_path}/delay_df_phase_2_flights.parquet")

### Logistic Regression Dummy Model Tester

In [0]:
# dtypes_dict = dict(df_otpw_engr_flights_new.dtypes)

# selected_flights_columns = [
#     c for c in df_otpw_engr_flights_new.columns
#     if c != "DEP_DEL15"
#     and c != "FL_DATE"
#     and dtypes_dict[c] != "string"
#     and "_idx" not in c
# ]

# features = selected_flights_columns
# # Drop rows with nulls in feature columns, then pass feature list directly
# df_clean = df_otpw_engr_flights_new.dropna(subset=features)

In [0]:
run_logistic_regression(
    df_scaled_flights,
    features="features",
    label_col="label",
    test_ratio=0.2,
    seed=42
)


## Stations Column Selection in OTPW

### Column Selector

In [0]:
stations_cols_string = [
    "STATION",
    "LATITUDE",
    "LONGITUDE"
]

stations_cols_numeric = [
]

stations_cols_categorical = [

]

In [0]:
selected_stations_columns = stations_cols_numeric + stations_cols_categorical + stations_cols_string
df_otpw_engr_stations = df_otpw_engr[selected_stations_columns + ["FL_DATE", "DEP_DEL15"]]
display(df_otpw_engr_stations)


### Clean and Cast Stations Columns

In [0]:
def cleanStationsColumns(df_otpw_engr_stations):
    # Clean and cast numeric stations columns
    for c in stations_cols_numeric:
        if c in df_otpw_engr_stations.columns:
            df_otpw_engr_stations = df_otpw_engr_stations.withColumn(c, expr(f"try_cast(`{c}` as double)"))

    # Clean and cast categorical stations columns
    for c in stations_cols_categorical:
        if c in df_otpw_engr_stations.columns:
            df_otpw_engr_stations = df_otpw_engr_stations.withColumn(c, expr(f"try_cast(`{c}` as string)"))
    
    return df_otpw_engr_stations


df_otpw_engr_stations = cleanStationsColumns(df_otpw_engr_stations)


### Feature Engineered Columns for Stations

In [0]:
created_stations_columns = []

selected_stations_columns = selected_stations_columns + created_stations_columns


### Logistic Regression Dummy Model Tester

In [0]:
# run_logistic_regression(df_otpw_engr_stations, features=selected_stations_columns, label_col="DEP_DEL15", test_ratio=0.2, seed=42)


## Final Feature Engineered Data Frame

### Data Frame Creation

In [0]:
final_engr_columns = selected_weather_columns + selected_flights_columns #+ selected_stations_columns

df_engr = df_otpw_engr[final_engr_columns + ["FL_DATE", "DEP_DEL15"]]
display(df_engr)


### Combined Data Processing

In [0]:

df_engr = weather_feature_engr(df_engr)
df_engr = cleanWeatherColumns(df_engr)
df_engr = impute_weather_columns(df_engr)

df_engr = cleanFlightsColumns(df_engr)
df_engr = featureEngineeredFlights(df_engr)

# df_engr = cleanStationsColumns(df_engr)


In [0]:
df_engr.printSchema()

In [0]:
# Checkpoint the weather + flights work so that we can use this for the further analysis in case everything breaks again

# Create folder
section = "3"
number = "1"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"
dbutils.fs.mkdirs(folder_path)

# Save df_etl as a parquet file
df_engr.write.mode('overwrite').parquet(f"{folder_path}/testing/df_engr_basic.parquet")


### Additional Feature Engineered Columns


###Normalization of final dataset numeric features

In [0]:
time_cols = ["FL_DATE", "YEAR", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "QUARTER", "CRS_DEP_TIME", "CRS_ARR_TIME"]
numeric_cols = [c for c, dtype in df_engr.dtypes if c not in time_cols and dtype in ['int', 'double', 'float', 'vector']]

In [0]:
numeric_cols

In [0]:
# Normalization of the numeric features of the df_engr_mini_sample dataset using StandardScaler
df_corr = df_engr.select(*(numeric_cols+time_cols+["DEP_DEL15"])).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vec = assembler.transform(df_corr)

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model = scaler.fit(df_vec)
df_scaled = scaler_model.transform(df_vec) #.select("scaled_features")

In [0]:
df_scaled.printSchema()

In [0]:
display(df_scaled.select("FL_DATE", "MONTH", "features", "scaled_features", "DEP_DEL15"), limit = 10)

In [0]:
# Assemble features

df_scaled_fin = df_scaled.drop("features") #I want to overwrite the features as it is already scaled in scaled_features
feature_cols = ["YEAR", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "QUARTER", "CRS_DEP_TIME", "CRS_ARR_TIME", "scaled_features"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
df_model = assembler.transform(df_scaled_fin).withColumn("label", col("DEP_DEL15").cast("double"))

df_model = df_model.select("FL_DATE", "MONTH", "features", "label")


### Testing Final Columns Potential MultiCollinearity

In [0]:
df_corr = df_engr.select(*(numeric_cols+["DEP_DEL15"])).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vec = assembler.transform(df_corr)

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model = scaler.fit(df_vec)
df_scaled = scaler_model.transform(df_vec).select("scaled_features")

corr_matrix = Correlation.corr(df_scaled, "scaled_features", "pearson") \
                         .collect()[0][0].toArray()

corr_df = pd.DataFrame(corr_matrix, columns=numeric_cols, index=numeric_cols)

#Plot correlation matrix
plt.figure(figsize=(10,8))
sns.heatmap(corr_df, annot=False, xticklabels=numeric_cols, yticklabels=numeric_cols, cmap="coolwarm")
plt.title("Correlation Matrix of Standardized Numeric Features")
plt.show()
plt.savefig("/tmp/FinalFeatureEngineeringCorrelationMatrix.png")

In [0]:
# Based off the correlation matrix, we will drop the:
# * HourlyDewPointTemperature - highly correlated with HourlyDryBulbTemperature
# * HourlyPressureTendency - highly correlated with HourlyHourlyPressureChange

# * Potential to remove HourlyWindGustSpeed - midly correlated with HourlyWindSpeed, but we'll keep it for now

df_engr_stdized = df_engr_stdized.drop("HourlyDewPointTemperature", "HourlyPressureTendency")
display(df_engr_stdized)


In [0]:
# Conversion of df to features and label


features = [c for c in df_numeric.columns if c != "DEP_DEL15" and c != "FL_DATE" and dict(df_numeric.dtypes)[c] != 'string']

# Assemble features
assembler = VectorAssembler(inputCols=features, outputCol="features", handleInvalid="skip")
df_model = assembler.transform(df_numeric).withColumn("label", col("DEP_DEL15").cast("double"))

df_model = df_model.select("FL_DATE", "MONTH", "features", "label")




### Logistic Regression Dummy Model Tester

In [0]:
df_model.printSchema()

features

In [0]:
df_model = df_model.dropna(subset="features")
run_logistic_regression(df_model, features="features", label_col="label", test_ratio=0.2, seed=42)

In [0]:
display(df_engr_stdized, limit=10)

display(df_numeric, limit = 10)

display(df_model, limit=10)


## Checkpoint Data

In [0]:
# Create folder
section = "3"
number = "1"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"
dbutils.fs.mkdirs(folder_path)

# Save df_etl as a parquet file
df_model.write.mode('overwrite').parquet(f"{folder_path}/delay_df_phase_2.parquet")


### Feature Engineering Pipeline

```mermaid
flowchart TD
    %% ── Data Loading ──
    A["📂 OTPW_12M_2015.csv.gz"] --> B["df_otpw\n(raw CSV load)"]
    B --> C["df_otpw_engr\nCast FL_DATE → timestamp\nCast DEP_DEL15 → boolean\nFilter nulls"]

    %% ── Three Parallel Branches ──
    C --> W1["🌦 Weather Column Selection\n14 numeric + 2 string cols"]
    C --> F1["✈️ Flights Column Selection\n3 categorical + 5 numeric + 7 string cols"]
    C --> S1["📡 Stations Column Selection\nSTATION, LATITUDE, LONGITUDE"]

    %% ── Weather Branch ──
    W1 --> W2["Weather Feature Engineering\n• ceiling_height_ft from HourlySkyConditions\n• has_thunder, has_rain, has_snow\n• has_fog, has_freezing, has_haze"]
    W2 --> W3["Clean & Cast Weather\n• Strip NOAA quality flags: *, s\n• T → 0, VRB → null\n• Cast to double"]
    W3 --> W4["Impute Weather Nulls\n• Zero-fill: WindGust, Precip, PressureChange\n• Domain-fill: ceiling → 99999\n• Median-fill: Temp, Humidity, Visibility"]
    W4 --> W5["🧪 Dummy LR Test\n(weather only)"]

    %% ── Flights Branch ──
    F1 --> F2["Clean & Cast Flights\n• Cast numeric cols to double/int\n• Cast categorical cols"]
    F2 --> F3["Flights Feature Engineering\n• departure_sequence\n• scheduled_flights_before_current\n• morning_delay_propagation\n• cum_delay_rate_today\n• 1d / 7d / 15d / 30d delay propagation\n• is_us_holiday, days_to_nearest_holiday\n• holiday_proximity_bucket"]
    F3 --> F5["🧪 Dummy LR Test\n(flights only)"]

    %% ── Stations Branch ──
    S1 --> S2["Clean & Cast Stations\n(partially implemented)"]

    %% ── Combine ──
    W4 --> MERGE["🔗 Merge All Columns\ndf_engr = weather + flights + stations"]
    F3 --> MERGE
    S2 -.-> MERGE

    %% ── Combined Processing ──
    MERGE --> CP1["Combined Processing\nweather_feature_engr → cleanWeather\n→ impute_weather → cleanFlights\n→ featureEngineeredFlights"]
    CP1 --> CK1["💾 Checkpoint\ndf_engr_basic.parquet"]

    %% ── Undersampling ──
    CK1 --> US["⚖️ Undersampling\n(optional — currently disabled)\nBalance DEP_DEL15 classes"]
    US --> CK2["💾 Checkpoint\ndf_engr_undersampled.parquet"]

    %% ── Normalization ──
    CK2 --> NORM["📊 StandardScaler Normalization\nZ-score numeric cols\n(excluding time cols)"]

    %% ── Multicollinearity ──
    NORM --> CORR["🔍 Correlation Matrix\nPearson on standardized features"]
    CORR --> DROP1["Drop Correlated Columns\n• HourlyDewPointTemperature\n• HourlyPressureTendency"]

    %% ── Final Model Prep ──
    DROP1 --> DROP2["Drop Non-Numeric Columns\nORIGIN, DEST, TAIL_NUM,\nOP_UNIQUE_CARRIER, state cols,\nstring propagation cols"]
    DROP2 --> ASM["🔧 VectorAssembler\ntime_cols + scaled_features → features\nlabel = DEP_DEL15"]
    ASM --> CK3["💾 Final Checkpoint\ndelay_df_phase_2.parquet"]
    CK3 --> LR["🧪 Logistic Regression\nDummy Model Test"]

    %% ── Styling ──
    classDef data fill:#e3f2fd,stroke:#1565c0,color:#0d47a1
    classDef process fill:#fff3e0,stroke:#e65100,color:#bf360c
    classDef test fill:#e8f5e9,stroke:#2e7d32,color:#1b5e20
    classDef checkpoint fill:#f3e5f5,stroke:#6a1b9a,color:#4a148c
    classDef merge fill:#fce4ec,stroke:#c62828,color:#b71c1c

    class A,B,C data
    class W1,W2,W3,W4,F1,F2,F3,S1,S2,NORM,CORR,DROP1,DROP2,ASM process
    class W5,F5,LR test
    class CK1,CK2,CK3 checkpoint
    class MERGE,CP1,US merge
```